# Quality Control (GWASQC)

Pipeline: https://github.com/MataLabCCF/GWASQC

**Version**: 1.0.1  
**Last iteration**: 22-MAR-2026  

## Imports


In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess
import numpy as np
import pandas as pd
from datetime import date

In [ ]:
d = date.today()

print(f'''
Last iteration: {d}

pandas=={pd.__version__}
numpy=={np.__version__}
''')

## Set directories and variables

### Common paths

In [ ]:
# Directories
# ~/.presentation/input/NeuroBooster
# Hestia NGS Software
tools = "/path/to/tools"

# Main directory
main_dir = "/path/to/analysis"
MAIN_DIR = main_dir     ### alias

# Data directory
data_dir = f"{main_dir}/input"
DATA_DIR = data_dir     ### alias

# Raw data directory
raw_dir = f"{data_dir}/raw"
RAW_DIR = raw_dir     ### alias

# Imputed data directory
impt_dir = f"{data_dir}/imputed"
IMPT_DIR = impt_dir     ### alias

# Meta data (covariate, population, ancestry labels, etc.)
meta_dir = f"{data_dir}/meta"
META_DIR = meta_dir     ### alias

### Paths to software and tools

In [ ]:
# Plink1.9 and Plink2.0 path
plink1 = f"{tools}/plink_linux_x86_64_20250615/plink"
plink2 = f"{tools}/plink2_linux_avx2_20250609/plink2"

# KING path
king = f"{tools}/king/king"

# Eagle
eagle = f"{tools}/eagle/Eagle_v2.4.1/eagle"

# Bcftools
bcftools = f"{tools}/bcftools-1.19/bcftools"
# Export bcftools plugins as env variable
os.environ["BCFTOOLS_PLUGINS"] = f"{tools}/bcftools-1.19/plugins"

# BGZIP
bgzip = f"{tools}/tabix-0.2.6/bgzip"

# GWASQC path
gwasqc = f"{tools}/GWASQC/main.py"

# XWAS path
xwas = f"{tools}/XWAS"
xwasqc = f"{xwas}/harmonizationAdapted.py"

# NAToRA path
natora = f"{tools}/NAToRA_Public/NAToRA_Public.py"

### Input, output, covariate files

In [ ]:
# Input file path without suffix 
rawFile = f"{raw_dir}/CATPD"
inputPfile = f"{raw_dir}/CATPD"      ### Updated IDs, Sex and added PD Pheno

# Covariate file
covar = f"{meta_dir}/CATPD_covariate_for_qc.cov"

# Populations/Ancestries file
pops = f"{meta_dir}/CATPD_covariate_for_qc.pop"

# Pheno Name
pheno = "STATUS"

# Keep file
keep = f"{meta_dir}/CATPD_covariate_for_qc.keep"

# Chromosomes as list
chromosomes = list(range(1,23))

# Output directory path and output prefix
outdir = f"{raw_dir}/QC_autosomes_X_chrom"
os.makedirs(outdir, exist_ok=True)
output = "CATPD"

### SNPs to exclude defined by GP2 and LARGE-PD

In [ ]:
# SNPs to exclude

# LARGEPD
snps_largepd = f"{meta_dir}/exclude.txt"

# GP2 underperforming
snps_gp2_underperf = f"{meta_dir}/underperforming_snps.txt"

### Reference genomes and other reference files

In [ ]:
### HG38

# FASTA
# Hestia NGS_Reference
reference_path = f"/path/to/reference"

# Hg38
fasta = f"{reference_path}/fasta/hg38/Homo_sapiens_assembly38.fasta"

# Genetic Map hg38
genmap = f"{xwas}/genetic_map_hg38_withX.txt"

# Structural regions to exclude
structural_regions = f"{xwas}/TriTyper.txt"

# OneThousand
onekg = f"{reference_path}/OneThousand/CCDG_14151_B01_GRM_WGS_2020-08-05_chrX.filtered.eagle2-phased.v2.vcf.gz"

# gnomAD files
gnomad = f"{reference_path}/gnomAD/gnomadOnlyAF_onePercent_*.vcf.gz"

# Threads by default
threads = 1

## Exclude GP2 underperforming SNPs and LARGE-PD SNPs

In [ ]:
def excludeSNPs(rawFile, inputPfile, snps_largepd, snps_gp2_underperf):
    snps_LARGEPD = ["plink2",
            "--pfile", rawFile,
            "--exclude", snps_largepd,
            "--keep", keep,
            "--make-pgen",
            "--out", f"{rawFile}.snps.largepd", 
    ]
    subprocess.run(snps_LARGEPD, check=True)
    
    snps_GP2 = ["plink2",
            "--pfile", f"{rawFile}.snps.largepd",
            "--exclude", snps_gp2_underperf,
            "--make-pgen",
            "--out", f"{rawFile}.snps.largepd.gp2",
    ]
    subprocess.run(snps_GP2, check=True)
    
    make_bed = ["plink2",
            "--pfile", f"{rawFile}.snps.largepd.gp2",
            "--make-pgen",
            "--out", inputPfile,
    ]
    subprocess.run(make_bed, check=True)

In [ ]:
excludeSNPs(rawFile, inputPfile, snps_largepd, snps_gp2_underperf)

## Subset raw array data into autosomes and X chromosome

Pipeline works with plink1.9 files. Convert plink2 pfiles to bfiles.  
Extract autosomes and X chromosome separately. Run QC on autosomes, then run XWAS QC on X chromosome.

In [ ]:
# Split input file
def split_by_chr(inputPfile, threads=1):
    
    makeBfile = ["plink2", 
                 "--pfile", inputPfile,
                 "--threads", str(threads),
                 "--make-bed", 
                 "--out", inputPfile
    ]

    ext = os.path.splitext(f"{inputPfile}.psam")[1]
    if ext in [".psam", ".pgen", ".pvar"]:
        subprocess.run(makeBfile)
    
    run_chrom_autosome = ["plink2", 
                 "--bfile", inputPfile,
                 "--chr", "1-22",
                 "--threads", str(threads),
                 "--make-bed", 
                 "--out", f"{inputPfile}.autosomes"
    ]
    subprocess.run(run_chrom_autosome, check=True)
    
    run_chrom_x = ["plink2", 
                 "--bfile", inputPfile,
                 "--chr", "X",
                 "--threads", str(threads),
                 "--make-bed", 
                 "--out", f"{inputPfile}.X"
    ]
    subprocess.run(run_chrom_x, check=True)

In [ ]:
split_by_chr(inputPfile)

## Quality control: Autosomes

### Run GWASQC pipeline

In [ ]:
run_gwasqc = ["python3.9", gwasqc,
        "--plink1", plink1,
        "--plink2", plink2,
        "--inputFile", inputPfile,
        "--info", covar,
        "--outputFolder", outdir,
        "--outputName", output,
        "--popFile", pops,
        "--savePerPop",
        "--info", covar,
        "--NAToRA", natora,
]

In [ ]:
subprocess.run(run_gwasqc, check=True)

#### Split autosomes and chrX

We're going to use autosomes (_HWECase) as input for chrX QC

In [ ]:
split_by_chr(f"{outdir}/{output}_HWECase")

## Quality control: X chromosome

### Extract samples from autosomes after QC in X chromosome

Use autosomes_HWECase subset of autosomal bfiles, because they contain related individuals afer QC.  
The FinalData directory contains QCed samples after related individuals are removed.

In [ ]:
# Split input file
def keep_qcd_samples(inputX, outputX, autosomal, threads=1):
    run_chrom_autosome = ["plink2", 
                 "--bfile", unqced_chrX,
                 "--keep", f"{autosomal}.fam",
                 "--threads", str(threads),
                 "--make-bed", 
                 "--out", filtered_chrX,
    ]
    subprocess.run(run_chrom_autosome, check=True)

In [ ]:
# Autosomal chr after QC
autosomal = f"{outdir}/CATPD_QC_HWECase.autosomes"

# ChrX unqced, qced, filtered
unqced_chrX = f"{inputPfile}.X"
filtered_chrX = f"{inputPfile}.qced_X"
qced_chrX = f"{output}_HWECase.X"

In [ ]:
keep_qcd_samples(unqced_chrX, filtered_chrX, autosomal)

### Run XWAS-QC pipeline

In [ ]:
run_xwasqc = ["python3", xwasqc,
              "--autosomal", autosomal,
              "--xchromosome", filtered_chrX,
              "--covar", covar, 
              "--phenotype", pheno,
              "--geneticMap", genmap, 
              "--structural", structural_regions,
              "--folder", outdir,
              "--output", qced_chrX,
              "--oneThousand", onekg, 
              "--gnomAD", gnomad,
              "--interpreter", "python3.8",
              "--plink", plink1,
              "--plink2", plink2,
              "--bcftools", bcftools,
              "--bgzip", bgzip,
              "--eagle", eagle,
              "--king", king,
              "--NAToRA", natora,
              "--threads", str(threads),
]

<div class="alert alert-block alert-info">
<b>Tip:</b> Includes phasing, takes one hour to run on Hestia
</div>

In [ ]:
subprocess.run(run_xwasqc, check=True)

## Preimputation check and subset into chromosomes

### Fix references with bcftools +fixref

In [ ]:
def prepVCF(qcd_raw, fasta):
    # df = pd.read_csv(f"{outdir}/{qced_chrX}_chrX_diffMAF.fam", sep='\s+', header=None, names=['FID', 'IID', 'PATID', 'MATID', 'SEX', 'PHENO'])
    # df['FID'] = 0
    # df[['FID', 'IID']].to_csv(f"{outdir}/{qced_chrX}_chrX_diffMAF.keep", sep='\t', index=False, header=False)
    
    exportVCF = ["plink2",
            "--bfile", qcd_raw,
            # "--keep", f"{outdir}/{qced_chrX}_chrX_diffMAF.keep",
            "--output-chr", "chr26",
            "--export", "vcf-iid", "bgz",
            "--out", qcd_raw, 
    ]
    subprocess.run(exportVCF, check=True)
    
    fixRef = ["bcftools",
            "+fixref", 
            f"{qcd_raw}.vcf.gz",
            "-Oz",
            "-o", f"{qcd_raw}.norm.vcf.gz",
            "--",
            "-f", fasta,
            "-m", "flip", 
            "-d"
    ]
    subprocess.run(fixRef, check=True)

In [ ]:
prepVCF(autosomal, fasta)

### Subset by chromosomes

#### Autosomes

In [ ]:
def subsetter(qcd_raw, preimpute):    
    subsetChr = ["plink2",
                "--vcf", f"{qcd_raw}.norm.vcf.gz",
                "--output-chr", "chr26",
                "--chr", str(c),
                "--recode", "vcf-iid", "bgz",
                "--out", preimpute,
                ]
    subprocess.run(subsetChr, check=True)

In [ ]:
preimpute_dir = f"{raw_dir}/QC"
os.makedirs(preimpute_dir, exist_ok=True)

In [ ]:
for c in chromosomes:
    preimpute = f"{preimpute_dir}/chr{c}.TOPMED"
    subsetter(autosomal, preimpute)

#### Chromosome X

In [ ]:
def fixRefX(inputXvcfgz, fasta, preimpute_dir=preimpute_dir):

    fixRef = ["bcftools",
            "+fixref", 
            f"{inputXvcfgz}.vcf.gz",
            "-Oz",
            "-o", f"{preimpute_dir}/chrX.TOPMED.vcf.gz",
            "--",
            "-f", fasta,
            "-m", "flip", 
            "-d"
    ]
    subprocess.run(fixRef, check=True)

In [ ]:
fixRefX(f"{outdir}/{qced_chrX}_chrX_Phased", fasta)

## Submit to TOPMed through API 

In [ ]:
import os, json
import requests
from contextlib import ExitStack

In [ ]:
# Imputaiton job name with current date (defined in the beginning of the notebook)
imputename = f"CATPD_{d}"
print(imputename)

In [ ]:
# imputation server url
BASE_URL = 'https://imputation.biodatacatalyst.nhlbi.nih.gov/api/v2'
AUTH_TOKEN = 'YOUR_API_TOKEN'

In [ ]:
preimpute_dir = '/path/to/preimpute'

Note: Read more in API Reference for TOPMed  
https://statgen.github.io/tis-v2-docs/api/api-reference/#__tabbed_1_2

In [ ]:
# add token to header (see documentation for Authentication)
headers = {'X-Auth-Token' : AUTH_TOKEN }
data = {
  'job-name': 'CATPD_Autosomes_PHASING',
  'refpanel': 'topmed-r3',
  'mode': 'phasing',
  'population': 'all',
  'build': 'hg38',
  'phasing': 'eagle',
  'r2Filter': 0,
  'password': 'YOUR_PASSOWRD'
}

# submit new job. This demonstrates multiple files, one per chromosome. Edit to send one or more chromosomes, as needed.
vcfs = [f"{preimpute_dir}/chr{c}.TOPMED.vcf.gz" for c in range(1, 23)]

with ExitStack() as stack:
    files = [
        ("files", stack.enter_context(open(vcf, "rb")))
        for vcf in vcfs
    ]

    ENDPOINT = "/jobs/submit/imputationserver2"
    resp = requests.post(
        BASE_URL + ENDPOINT,
        files=files,
        data=data,
        headers=headers
    )

output = resp.json()

if resp.status_code != 200:
  print(output['message'])
  raise Exception('POST {} {}'.format(endpoint, resp.status_code))
else:
    # print message
    print(output['message'])
    print(output['id'])

## Download imputed .zip files after completion

Set the current downloaded area as a working directory

In [ ]:
WD = f"{impt_dir}/IMPUTED_ALL_AUTOSOME_CHRX"

In [ ]:
print(WD)

In [ ]:
os.chdir(WD)

### Unzip files with password

<div class="alert alert-block alert-info">
⚠️ Best way to run this is use bash command with nohup for all chr in parallel
</div>

In [ ]:
%%bash

password="YOUR_PASSWORD"
unzipped_dir="/path/to/unzipped"

for c in {1..22}; do echo -e "nohup unzip -P ${password} chr_${c}.zip -d ${unzipped_dir} > chr${c}.log 2>&1 &" ; done

### Extract plink files, softcalls and QC data

In [ ]:
def imputedQualityControl(vcfPrefix=None, rawPlinkPsam=None, inputPath=None, plinkPrefix=None,
                          pheno="STATUS", r2=0.3, maf=0.05, mac=10, hwe=1e-5,
                          update_varids="@:#:$r:$a", update_varids_multi="@:#:$1:$2_multi", 
                          update_varids_nonsnp="@:#_multi_$1_$2", max_allele_len=100, threads=1):
    # Set inputPath as current work directory if not set 
    if inputPath is None:
        inputPath = os.getcwd()
    # Check if vcf prefix is provided
    if vcfPrefix is None:
        raise ValueError("Provide vcf file prefix") 
    # Check if plink2 prefix is provided
    if plinkPrefix is None:
        plinkPrefix = vcfPrefix
    # Check if raw plink file is provided
    if rawPlinkPsam is None:
        raise ValueError("Provide raw file psam with sex and phenotype information")  
        
    # Create directories for data
    for d in ["plink_noqc", "softcalls", "plink_qc"]:
        new_dir = os.path.join(inputPath, d)
        os.makedirs(new_dir, exist_ok=True)

    # Set directories
    plink_noqc = f"{inputPath}/plink_noqc"
    plink_qc = f"{inputPath}/plink_qc"
    softcalls = f"{inputPath}/softcalls"

    # For vcfPrefix do not put .vcf.gz 
    # Change variant ids chr:pos:a1:a2
    inputVCF = f"{inputPath}/{vcfPrefix}.vcf.gz"
    plinkTemp = f"{plink_noqc}/{plinkPrefix}_temp"

    # Unzip downloaded files
    # unzipper = ["unzip", "-P", password, vcfPrefix.zip, "-d", WD]
    # subprocess.run(unzipper)
    
    # Function
    VCFToPlink2 = [
        "plink2",
        "--vcf", inputVCF,
        "--psam", rawPlinkPsam,
        "--make-pgen",
        "--sort-vars",
        "--silent",
        "--threads", str(threads),
        "--out", plinkTemp,
    ]
    subprocess.run(VCFToPlink2, check=True)

    # Update SNP encoding
    plinkUpdSNP = f"{plink_noqc}/{plinkPrefix}"
    # Function
    fixSNP = [
        "plink2",
        "--pfile", plinkTemp,
        "--set-all-var-ids", update_varids,
        "--var-id-multi", update_varids_multi,
        "--var-id-multi-nonsnp", update_varids_nonsnp,
        "--new-id-max-allele-len", str(max_allele_len),
        "--make-pgen",
        "--sort-vars",
        "--silent",
        "--threads", str(threads),
        "--out", plinkUpdSNP,
    ]
    subprocess.run(fixSNP, check=True)
    
    # Extract softcalls from plink
    plinkSoft = f"{softcalls}/{plinkPrefix}"    
    # Function
    getSoftcalls = [
        "plink2",
        "--pfile", plinkUpdSNP,
        "--extract-if-info", f"R2>={r2}",
        "--make-pgen",
        "--silent",
        "--threads", str(threads),
        "--out", plinkSoft,
    ]
    subprocess.run(getSoftcalls, check=True)

    # QC the imputed data for GWAS
    plinkQC = f"{plink_qc}/{plinkPrefix}"    
    # Function
    plink_qc = [
        "plink2",
        "--pfile", plinkSoft,
        "--maf", str(maf),
        "--mac", str(mac),
        "--hwe", str(hwe),
        "--make-pgen",
        "--silent",
        "--threads", str(threads),
        "--out", plinkQC,
    ]
    subprocess.run(plink_qc, check=True)

In [ ]:
rawPlinkPsam = f"{WD}/CATPD_related.psam"

In [ ]:
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(
            imputedQualityControl,
            f"chr{c}.dose",
            rawPlinkPsam
        )
        for c in range(1, 23)
    ]
    for fut in as_completed(futures):
        try:
            fut.result()
        except Exception as e:
            print(e)